# LAB 07 — Transformer Classification Experiments

Notebook này triển khai Bài 19–23, 25 và 29 theo W7.pdf. Mọi metrics/đồ thị được tạo từ model thực sự chạy. Các phần nhận xét cá nhân để sinh viên tự hoàn thiện.

## 1. Imports & Environment

Import TransformerBlock tự cài ở Bài 17; chạy regression tests để chắc thay đổi mask/inspection vẫn giữ hoạt động cũ.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import torch
from IPython.display import display

LAB_DIR = Path.cwd()
if not (LAB_DIR / "W7.pdf").exists():
    LAB_DIR = (LAB_DIR / "Lab7").resolve()
sys.path.insert(0, str(LAB_DIR))

from transformer_block import TransformerBlock, run_tests as test_transformer_block
from transformer_experiments import (
    ExperimentRunner, SEED, SPLIT_SEED, D_MODEL, D_FF, BATCH_SIZE,
    LEARNING_RATE, EPOCHS, MAX_LENGTH, MAX_LENGTHS,
)

test_transformer_block()

## 2. Configuration & Reproducibility

`full` dùng toàn bộ split LAB5 và 5 epochs. `smoke` chỉ kiểm tra pipeline trên các tập con xác định (2.048 train, 512 validation, 1.024 test) với 1 epoch; không dùng smoke metrics thay cho kết quả full.

In [ ]:
RUN_MODE = "full"  # đổi thành "smoke" để kiểm tra nhanh pipeline
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CONFIG = {
    "seed": SEED,
    "split_seed_lab5": SPLIT_SEED,
    "d_model": D_MODEL,
    "d_ff": D_FF,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "epochs": EPOCHS if RUN_MODE == "full" else 1,
    "max_length_default": MAX_LENGTH,
    "max_lengths": list(MAX_LENGTHS),
    "transformer_blocks": 1,
    "run_mode": RUN_MODE,
    "device": DEVICE,
}
print(CONFIG)
print("CUDA available:", torch.cuda.is_available())

## 3. Dataset Loading — IMDb, đúng split LAB5

Dùng split IMDb có nhãn `train` và `test`; không dùng `unsupervised`. IMDb train được chia 80/20 stratified với `random_state=2026` như LAB5; test gốc giữ nguyên. Model seed là 42.

In [ ]:
runner = ExperimentRunner(LAB_DIR, device=DEVICE, run_mode=RUN_MODE)
ENVIRONMENT = runner.environment_summary()
display(pd.Series(ENVIRONMENT, name="Value").to_frame())
print("Train/validation/test sizes:", len(runner.train_ids), len(runner.val_ids), len(runner.test_labels))
print("Vocabulary size, including PAD/UNK:", len(runner.vocab))
print("Label mapping: 0=Negative, 1=Positive; positive class for F1=1")

## 4. Tokenizer & Vocabulary

Tái sử dụng cách làm sạch/tokenizer regex LAB5. Vocabulary tối đa 20.000 mục, min frequency 2, chỉ dựng từ train. PAD và UNK có ID riêng. Review được truncate bên phải theo max length và padding động theo batch.

In [ ]:
print("Tokenizer: HTML unescape + lowercase + strip HTML + [a-z0-9]+(?:'[a-z]+)?")
print("PAD_ID:", 0, "UNK_ID:", 1)
print("Vocabulary sample:", list(runner.vocab.items())[:12])
for split, ids, labels in (
    ("train", runner.train_ids, runner.train_labels),
    ("validation", runner.val_ids, runner.val_labels),
):
    print(split, "samples:", len(ids), "labels:", pd.Series(labels).value_counts().sort_index().to_dict())

## 5. Shared Training & Evaluation Utilities

`ExperimentRunner` dùng chung loaders, optimizer Adam, CrossEntropyLoss, metrics binary (positive=1), early checkpoint selection theo Validation F1 và loss trung bình theo số samples. Test chỉ được gọi ở phần final evaluation.

In [ ]:
print("Batch size:", BATCH_SIZE, "| LR:", LEARNING_RATE, "| epochs:", CONFIG["epochs"])
print("Attention masks exclude PAD keys; pooling excludes PAD positions.")
print("Train shuffling restarts with the same seed for each independent model.")

## 6. Transformer Classifier — Bài 23

Pipeline: Embedding → sinusoidal positional encoding → một TransformerBlock tự cài → masked mean pooling → Linear classifier. Classifier trả logits; xác suất chỉ tính khi cần prediction. Có thể bật learned `[CLS]` vector ở Bài 25.

In [ ]:
print("TransformerBlock source:", (LAB_DIR / "transformer_block.py").resolve())
print("Padding mask is applied to attention keys and to mean pooling.")
print("One block; d_model=128, d_ff=512, dropout=0.1.")

## 7. Baseline Classification & 8. Number of Attention Heads — Bài 23 và 19

Mỗi số head tạo model mới, seed/optimizer/data/batch/epochs/length/pooling giữ nguyên. Bảng dùng Validation metrics để so sánh và chọn checkpoint; cột parameters là toàn classifier và có thêm số riêng của TransformerBlock.

In [ ]:
heads_results = runner.run_heads()
display(heads_results.style.format({
    "Training time (s)": "{:.2f}", "Validation accuracy": "{:.4f}",
    "Validation F1": "{:.4f}", "Best validation loss": "{:.4f}",
}))

## 9. Attention Head Diversity — Bài 20

Lấy attention weights từ model 4-head đã train, ở `eval()` mode. Heatmaps dùng chung scale; thống kê entropy, max weight và cosine similarity chỉ mô tả phân phối, không tự chứng minh ý nghĩa ngôn ngữ.

In [ ]:
head_stats, head_similarity = runner.run_head_diversity()
display(head_stats.style.format({
    "Mean attention entropy": "{:.4f}", "Mean maximum weight": "{:.4f}"
}))
display(head_similarity.style.format("{:.4f}"))

## 10. Transformer vs LSTM — Bài 21

LSTM dùng packed sequences để bỏ padding; hai model dùng chung data split, vocabulary, embedding/hidden size 128, batch, optimizer, LR, epochs và max length 128. Số parameters không ép bằng nhau. Bảng dưới là Validation; test sẽ được báo cáo một lần ở phần cuối.

In [ ]:
comparison_validation = runner.run_lstm()
display(comparison_validation.style.format({
    "Training time (s)": "{:.2f}", "Validation accuracy": "{:.4f}",
    "Validation F1": "{:.4f}", "Best validation loss": "{:.4f}",
}))

## 11. Sequence Length — Bài 22

So sánh max_length 32/64/128/256. Length 128 tái sử dụng đúng model 4-head đã train ở Bài 19. Tokens retained được đo trên train; attention memory là ước lượng cho **một** score/weight matrix `(batch, heads, T, T)`, không phải tổng bộ nhớ training.

In [ ]:
length_results = runner.run_sequence_lengths()
display(length_results.style.format({
    "Mean tokens retained/sample": "{:.2f}", "Train tokens retained (%)": "{:.2f}%",
    "Training time (s)": "{:.2f}", "Validation F1": "{:.4f}",
    "Validation accuracy": "{:.4f}",
}))

## 12. Pooling Strategy — Bài 25

Mean pooling loại padding. CLS-style dùng một learned `[CLS]` vector đặt ở đầu sequence; positional encoding/mask được dịch và cập nhật tương ứng. Chỉ thay pooling và token cần thiết.

In [ ]:
pooling_results = runner.run_pooling()
display(pooling_results.style.format({
    "Training time (s)": "{:.2f}", "Validation accuracy": "{:.4f}",
    "Validation F1": "{:.4f}",
}))

## 13. Learning Curves — Bài 29

Loss được lưu từng epoch, train loss theo train samples và validation loss theo validation samples. Biểu đồ không tự gán kết luận overfitting/underfitting.

In [ ]:
learning_history = runner.save_learning_curves()
display(learning_history.style.format({
    "train_loss": "{:.4f}", "validation_loss": "{:.4f}", "validation_f1": "{:.4f}"
}))

## 14. Confusion Matrix & Final Test Evaluation

Sau khi cấu hình chính được chọn qua Validation, đánh giá Transformer 4-head mean baseline và LSTM trên IMDb test split gốc (hoặc test subset nếu RUN_MODE=smoke). Confusion matrix tương ứng Transformer; tổng ô phải bằng số test samples.

In [ ]:
test_results, test_labels, transformer_test_predictions = runner.final_test_evaluation()
display(test_results.style.format({
    "Training time (s)": "{:.2f}", "Accuracy": "{:.4f}",
    "Precision": "{:.4f}", "Recall": "{:.4f}", "F1": "{:.4f}",
}))
print("Evaluated test samples:", len(test_labels))

## 15. Application — Năm loại input

Đây là kiểm tra hành vi trên câu thủ công, không thay evaluation test. Xác suất lấy softmax từ logits của model đã train.

In [ ]:
example_predictions = runner.predict_examples()
display(example_predictions.style.format({"P(Negative)": "{:.4f}", "P(Positive)": "{:.4f}"}))

## 16. Export Results

`results.csv` có các dòng riêng cho từng experiment, validation/test split và epoch loss. Trường không áp dụng để trống; kết quả smoke được đánh dấu `run_mode=smoke`.

In [ ]:
results = runner.export_results()
display(results.groupby(["experiment", "eval_split"], dropna=False).size().rename("rows").to_frame())
print("Results path:", LAB_DIR / "results.csv")
print("Images:", LAB_DIR / "attention_heads.png", LAB_DIR / "learning_curves.png", LAB_DIR / "confusion_matrix.png")

## 17. Student Interpretation TODOs

- Bài 19: Tăng head có tăng representation dimension không? `d_k` đổi ra sao? Vì sao F1 có thể không tăng?
- Bài 20: Head nào tập trung/phân tán? Các ma trận có giống nhau không? Evidence nào hỗ trợ nhận xét?
- Bài 21: Khác biệt architecture nào có thể liên quan đến kết quả; sequential và parallel processing khác gì?
- Bài 22: Vì sao không chỉ tăng sequence length? T² và thời gian thực tế thay đổi ra sao?
- Bài 25: Pooling ảnh hưởng gì tới representation và kết quả trên split này?
- Bài 29: Quan sát convergence/instability/overfitting từ từng đường loss; không kết luận khi chưa có bằng chứng.

## 18. AI Assistance Statement — sinh viên rà soát và hoàn thiện

```text
Tool: OpenAI Codex
Purpose: hỗ trợ boilerplate, debugging, visualization và chạy/kiểm tra experiments.
Generated: [liệt kê đúng phần code/biểu đồ được AI hỗ trợ]
Modified: [ghi phần em đã tự sửa]
Verified: [ghi những phép kiểm tra em đã tự xác nhận]
```

Notebook không viết thay phần interpretation, error analysis hoặc reflection cá nhân.